# 03 — Waste Reduction: Simulation & Recommendations

**Business question:** How much waste (and money) can FreshMart save with smarter ordering and earlier markdowns?

**What we learned in 02:**
- Meat waste: 18.2% avg; weekday ~20% vs weekend ~13%
- Weekday over-order gap (~47 units) is much larger than weekend (~29)
- Markdowns lose money (−17% margin) — applied too late to at-risk items
- Waste cost $3.5M = 45% of meat profit ($7.8M)

This notebook:
1. Simulates weekday order-quantity optimization
2. Tests earlier-markdown timing
3. Quantifies dollar impact and writes recommendations

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

df = pd.read_csv('/content/drive/MyDrive/포트폴리오-FreshMart(캐글)/clean_freshmart_data.csv')
meat = df[df['category'] == 'Meat'].copy()
print(f'Meat records: {len(meat):,}')

## 1. Baseline: current weekday performance

We restate the problem in dollars per weekday so the simulation has a clear target.

In [ ]:
base = meat.groupby('day_of_week').agg(
    n=('record_id', 'count'),
    avg_order=('initial_quantity', 'mean'),
    avg_sold=('units_sold', 'mean'),
    avg_waste_pct=('waste_pct', 'mean'),
    total_waste_cost=('waste_cost', 'sum'),
    total_profit=('profit', 'sum'),
)
base.index = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
print(base.round(1))

weekday_waste_cost = base.loc[['Mon','Tue','Wed','Thu','Fri'], 'total_waste_cost'].sum()
print(f'\nWeekday waste cost: ${weekday_waste_cost:,.0f}')

## 2. Simulation: right-size weekday orders

**Logic:** Weekend achieves 13% waste with an over-order buffer of ~29 units above sales. We simulate cutting weekday orders so the buffer matches the weekend level — i.e., order = avg sold + 29 units — instead of the current ~47-unit buffer.

**Assumption (stated):** Demand is unchanged; only the order quantity moves. Waste falls proportionally to the smaller buffer. This is a conservative, explainable estimate — not a forecast.

In [ ]:
weekend_buffer = 29  # units above avg sold (Sat/Sun average)

sim = base.copy()
weekdays = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri']

for d in weekdays:
    sold = sim.loc[d, 'avg_sold']
    new_order = sold + weekend_buffer
    # waste scales with buffer size: new_waste_pct = old * (new_buffer / old_buffer)
    old_buffer = sim.loc[d, 'avg_order'] - sold
    sim.loc[d, 'sim_order'] = new_order
    sim.loc[d, 'sim_waste_pct'] = sim.loc[d, 'avg_waste_pct'] * (weekend_buffer / old_buffer)

print(sim.loc[weekdays, ['avg_order', 'sim_order', 'avg_waste_pct', 'sim_waste_pct']].round(1))

# Dollar impact: waste cost falls proportionally to waste pct
sim['waste_saving'] = 0.0
for d in weekdays:
    reduction = 1 - sim.loc[d, 'sim_waste_pct'] / sim.loc[d, 'avg_waste_pct']
    sim.loc[d, 'waste_saving'] = sim.loc[d, 'total_waste_cost'] * reduction

total_saving = sim.loc[weekdays, 'waste_saving'].sum()
print(f'\nEstimated annual waste-cost saving: ${total_saving:,.0f}')
print(f'As % of meat waste cost: {total_saving/meat["waste_cost"].sum()*100:.1f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(7)
w = 0.35
labels = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
current = base['avg_waste_pct'].values
simulated = sim['sim_waste_pct'].fillna(base['avg_waste_pct']).values

ax.bar(x - w/2, current, w, label='Current', color='indianred')
ax.bar(x + w/2, simulated, w, label='Simulated (right-sized orders)', color='steelblue')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Waste %')
ax.set_title('Meat Waste % by Weekday: Current vs Simulated')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Markdown timing

**Logic:** In 02, markdown items showed −17% margin. We check *when* markdowns are applied — if they happen with almost no shelf life left, the discount is desperation, not strategy.

In [ ]:
md_items = meat[meat['markdown_applied'] == 1]
print(f'Markdown items: {len(md_items):,}')
print(md_items['days_until_expiry'].describe().round(1))

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(md_items['days_until_expiry'], bins=30, color='darkorange', edgecolor='white')
ax.set_xlabel('Days Until Expiry (at markdown)')
ax.set_ylabel('Items')
ax.set_title('When Are Markdowns Applied? (Meat)')
plt.show()

**Read it like a manager:** If most markdowns cluster at 0–1 days to expiry, the policy should move the trigger earlier (e.g., 3 days) with a smaller discount — protecting margin while still moving volume.

## 4. Recommendations

| # | Action | Expected effect |
|---|--------|----------------|
| 1 | Right-size weekday orders to weekend buffer levels | Cut weekday waste ~35%, save ~$X (see simulation) |
| 2 | Move markdown trigger earlier (3 days to expiry) with smaller discounts | Reduce −17% margin bleed on markdown items |
| 3 | Tighten cold-chain checks on high temp-deviation deliveries | Attack the spoilage driver found in 02 |
| 4 | Track waste % by weekday weekly — the KPI for this project | Make the improvement visible |

*Fill in $X with the simulation output above.*

## Interview one-liners

- "I found weekday over-ordering drove 20% waste vs 13% on weekends, so I simulated right-sizing weekday orders to weekend buffer levels."
- "Markdowns had −17% margin because they were applied too late — the fix is earlier triggers with smaller discounts, not deeper last-minute cuts."
- "Every cleaning decision is documented with reasoning in 01, so the analysis is reproducible."